# Configuratie Agent

De configuratie Agent van 'Talk with FEWS' heeft verschillende tools tot zijn beschikking.
Het taalmodel kan deze zelf invullen, maar hier wordt per tool een voorbeeld gegeven van de in en output. 

```python
# TODO: 
from talk_with_fews.tools.config.deadpoint_analysis import analyze_dead_points
from talk_with_fews.tools.config.dependency_analysis import analyze_dependencies
from talk_with_fews.tools.config.graph_visualize import visualize_graph
from talk_with_fews.tools.config.transformation_search import search_transformations
from talk_with_fews.tools.config.workflow_hierarchy import find_workflow_call_hierarchy
```

### Voorbereidingen
In eerste instantie worden de juiste variable opgehaald. Het context object bevat informatie over welke configuratie gebruikt wordt. In de voorbereiding is de configuratie al ingelezen in een database object. Dit is anders dan de normale 'FEWS database' waar alle data in zit. In deze notebook is alles met 'database' dus de configuratie database.


In [1]:
from pydantic_ai import format_as_xml
from setup_quarto import setup_dot_env

ctx = setup_dot_env()

c:\Data\Python\PR5473.10_Talk_with_Fews\talk-with-fews\.pixi\envs\default\Lib\site-packages\pydantic\fields.py:1042: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  warn(


# Search workflows
Workflows is een logisch start punt om de verschillende tools uit te leggen. 
Gegeven een (deel) van de workflow naam kan de informatie achter deze workflow worden opgehaald.
Ik weet dat ik een workflow wil hebben die begint met `im.matroos.rws.hist `, dus ik zoek deze workflow die deze naam bevatten.

In [2]:
name_pattern = "im.matroos.rws.hist"

In [3]:
from talk_with_fews.tools.config.workflow_search import search_workflows
from talk_with_fews.models.workflow_search import (
    WorkflowSearchOutput,
    WorkflowSearchInput,
)

workflow_search_input = WorkflowSearchInput(name_pattern=name_pattern)
workflow_search_output: WorkflowSearchOutput = await search_workflows(
    ctx=ctx, input=workflow_search_input
)

In [4]:
print(format_as_xml(workflow_search_output, root_tag="WorkflowDetailsOutput"))

<WorkflowDetailsOutput>
  <results>
    <example>
      <id>3</id>
      <wf_id>wf.im.matroos.rws.hist</wf_id>
      <desc>wf.im.matroos.rws.hist</desc>
      <module_config_ids>
        <example>135</example>
      </module_config_ids>
    </example>
  </results>
  <markdown_content>## Workflow Search Results
Found 1 workflows matching the pattern 'im.matroos.rws.hist'.

| ID | Workflow ID | Description | Module Config IDs |
|----|-------------|-------------|-------------------|
| 3 | wf.im.matroos.rws.hist | wf.im.matroos.rws.hist | 135 |
</markdown_content>
</WorkflowDetailsOutput>


## Get workflow details
Gegeven een workflow ID (bijvoorbeeld uit `search_workflows`) kan de bijbehorende informatie over deze workflow worden opgehaald.

In [5]:
workflow_id = workflow_search_output.results[0].id
workflow_id

3

In [6]:
from talk_with_fews.tools.config.workflows_details import get_workflow_details
from talk_with_fews.models.workflow_details import (
    WorkflowDetailsOutput,
    WorkflowDetailsInput,
)

workflow_details_input = WorkflowDetailsInput(workflow_id=workflow_id)
workflow_details_output: WorkflowDetailsOutput = await get_workflow_details(
    ctx=ctx, input=workflow_details_input
)

In [7]:
print(format_as_xml(workflow_details_output, root_tag="WorkflowDetailsOutput"))

<WorkflowDetailsOutput>
  <results>
    <example>
      <id>3</id>
      <wf_id>wf.im.matroos.rws.hist</wf_id>
      <desc>wf.im.matroos.rws.hist</desc>
      <wffn>wf.im.matroos.rws.hist</wffn>
      <module_calls>
        <example>
          <mic_id>135</mic_id>
          <modins>im.matroos.rws.hist</modins>
          <ord_nr>0</ord_nr>
        </example>
      </module_calls>
    </example>
  </results>
  <markdown_content>## Workflow: wf.im.matroos.rws.hist
**Description:** wf.im.matroos.rws.hist

### Basic Information
| Property | Value |
|----------|--------|
| ID | 3 |
| Function Name | wf.im.matroos.rws.hist |

### Module Calls and Their Transformations

#### Module: im.matroos.rws.hist
The following transformations are executed in this module:

| Order | Transformation ID | Description |
|-------|------------------|-------------|
| 1 | 182 | im.matroos.rws.hist |

### Workflow Structure
```mermaid
graph TD
    WF[wf.im.matroos.rws.hist]:::workflow
    M135[im.matroos.rws.hist]

In [8]:
print(workflow_details_output)

results=[WorkflowDetails(id=3, wf_id='wf.im.matroos.rws.hist', desc='wf.im.matroos.rws.hist', wffn='wf.im.matroos.rws.hist', module_calls=[ModuleInstanceCall(mic_id=135, modins='im.matroos.rws.hist', ord_nr=0)])] markdown_content='## Workflow: wf.im.matroos.rws.hist\n**Description:** wf.im.matroos.rws.hist\n\n### Basic Information\n| Property | Value |\n|----------|--------|\n| ID | 3 |\n| Function Name | wf.im.matroos.rws.hist |\n\n### Module Calls and Their Transformations\n\n#### Module: im.matroos.rws.hist\nThe following transformations are executed in this module:\n\n| Order | Transformation ID | Description |\n|-------|------------------|-------------|\n| 1 | 182 | im.matroos.rws.hist |\n\n### Workflow Structure\n```mermaid\ngraph TD\n    WF[wf.im.matroos.rws.hist]:::workflow\n    M135[im.matroos.rws.hist]:::module\n    WF --> M135\n    T182[im.matroos.rws.hist]:::transformation\n    M135 --> T182\n\n    classDef workflow fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px\n    classDef

## Get transformation details
Gegeven de transformatie ID kunnen we kijken wat verder met deze workflow te maken heeft. Bijvoorbeeld uit tabel in de `get_workflow_details` tool. 

In [9]:
transformation_id = 182

In [10]:
from talk_with_fews.tools.config.transformation_details import (
    get_transformation_details,
)
from talk_with_fews.models.transformation_details import (
    TransformationDetailsInput,
    TransformationDetailsOutput,
)

transformation_details_input = TransformationDetailsInput(
    transformation_id=transformation_id
)
transformation_details_output: TransformationDetailsOutput = (
    await get_transformation_details(ctx=ctx, input=transformation_details_input)
)

In [11]:
print(
    format_as_xml(transformation_details_output, root_tag="TransformationDetailsOutput")
)

<TransformationDetailsOutput>
  <results>
    <example>
      <id>182</id>
      <desc>im.matroos.rws.hist</desc>
      <ord_nr>1</ord_nr>
      <config_file>/ModuleConfigFiles/Import/im.matroos.rws.hist.xml</config_file>
      <mic_id>135</mic_id>
      <modins>im.matroos.rws.hist</modins>
      <time_series_links>
        <example>
          <inout>out</inout>
          <tss_id>14</tss_id>
        </example>
      </time_series_links>
    </example>
  </results>
  <markdown_content>## Transformation Details
**ID:** 182
**Description:** im.matroos.rws.hist
**Config File:** /ModuleConfigFiles/Import/im.matroos.rws.hist.xml

### Basic Information
| Property | Value |
|----------|--------|
| Order Number | 1 |
| Module Instance | im.matroos.rws.hist |
| Module Config ID | 135 |

### Time Series Connections
This transformation processes the following time series:

#### Input Time Series
No input time series connected.

#### Output Time Series
| ID | Description |
|----|-------------|
| 14

# Describe Timeseries
Gegeven een timeseries ID kan je bijbehorende informatie over de transformaties ophalen waar deze timeseries in voor komt. 
Hier gebruiken we een timeseries ID uit `get_transformation_details`. 

In [12]:
timeseries_id = transformation_details_output.results[0].time_series_links[0].tss_id
timeseries_id

14

In [13]:
from talk_with_fews.tools.config.timeseries_describe import describe_timeseries
from talk_with_fews.models.timeseries_describe import (
    DescribeTimeSeriesInput,
    DescribeTimeSeriesOutput,
)

timeseries_describe_input = DescribeTimeSeriesInput(timeseries_id=timeseries_id)
timeseries_describe_output: DescribeTimeSeriesOutput = await describe_timeseries(
    ctx=ctx, input=timeseries_describe_input
)

In [14]:
print(format_as_xml(timeseries_describe_output, root_tag="DescribeTimeSeriesOutput"))

<DescribeTimeSeriesOutput>
  <results>
    <example>
      <id>14</id>
      <modins>im.matroos.rws.hist</modins>
      <vtype>scalar</vtype>
      <parid>H.B</parid>
      <qualifiers />
      <tstype>external historical</tstype>
      <timess>day</timess>
      <description>im.matroos.rws.hist•scalar•H.B••external historical•day</description>
      <endpoint>False</endpoint>
      <endpoint_desc>null</endpoint_desc>
      <transformations />
    </example>
  </results>
  <markdown_content>## Time Series 14
**Description:** im.matroos.rws.hist•scalar•H.B••external historical•day

### Properties
| Property | Value |
|----------|--------|
| Module Instance | im.matroos.rws.hist |
| Value Type | scalar |
| Parameter ID | H.B |
| Qualifiers |  |
| Time Series Type | external historical |
| Time Step | day |

### Transformations
No transformations found</markdown_content>
</DescribeTimeSeriesOutput>


## Search Timeseries
Gegeven een transformatie kunnen we een tijdserie ophalen. Hier moet je wel alle informatie over de transformatie weten. 
Voor het gemak gebruikten we timeseries uit `describe_timeseries`

In [15]:
from talk_with_fews.models.base import TimeSeriesProperties


timeseries: TimeSeriesProperties = timeseries_describe_output.results[0]
print(format_as_xml(timeseries, root_tag="TimeSeriesProperties"))

<TimeSeriesProperties>
  <id>14</id>
  <modins>im.matroos.rws.hist</modins>
  <vtype>scalar</vtype>
  <parid>H.B</parid>
  <qualifiers />
  <tstype>external historical</tstype>
  <timess>day</timess>
  <description>im.matroos.rws.hist•scalar•H.B••external historical•day</description>
  <endpoint>False</endpoint>
  <endpoint_desc>null</endpoint_desc>
  <transformations />
</TimeSeriesProperties>


In [16]:
# from talk_with_fews.tools.config.timeseries_search import search_timeseries
# from talk_with_fews.models.timeseries_search import (
#     TimeSeriesSearchResult,
#     TimeSeriesSearchOutput,
# )

# timeseries = {'id': 14,
#  'modins': 'im.matroos.rws.hist',
#  'vtype': 'scalar',
#  'parid': 'H.B',
#  'qualifiers': '',
#  'tstype': 'external historical',
#  'timess': 'day',
#  'description': 'im.matroos.rws.hist•scalar•H.B••external historical•day',
#  'endpoint': False,
#  'endpoint_desc': None,
#  'transformations': []}
# # timeseries_search_input = TimeSeriesSearchResult(**timeseries.model_dump())
# timeseries_search_input = TimeSeriesSearchResult(**timeseries)

# timeseries_search_output: TimeSeriesSearchOutput = await search_timeseries(
#     ctx=ctx, input=timeseries_search_input
# )

# # TODO: AttributeError: 'TimeSeriesSearchResult' object has no attribute 'parameter_id'
# # This is correct as the `parameter_id` is not part of the `TimeSeriesProperties` model, its id..

## Get database schema
Om op de juiste manier de tool execute query te gebruiken, is het handig om te weten wat er in de database zit.
Hiervoor zijn de schema's van de configuratie database.
Deze tool geeft een overzicht van de beschikbare tabellen en kolommen.

In [17]:
from talk_with_fews.tools.config.get_database_schema import get_database_schema
from talk_with_fews.models.database_schema import (
    DatabaseSchemaOutput,
    DatabaseSchemaInput,
)

database_schema_input = DatabaseSchemaInput()

database_schema_output: DatabaseSchemaOutput = await get_database_schema(
    ctx=ctx, input=database_schema_input
)

In [18]:
print(format_as_xml(database_schema_output, root_tag="DatabaseSchemaOutput"))

<DatabaseSchemaOutput>
  <tables>
    <call_stacks>
      <example>id</example>
      <example>wf_id</example>
      <example>desc</example>
      <example>wffn</example>
      <example>parent_id</example>
      <example>props_set_id</example>
    </call_stacks>
    <endpoints>
      <example>tss_id</example>
      <example>desc</example>
    </endpoints>
    <links>
      <example>trafo_id</example>
      <example>tss_id</example>
      <example>inout</example>
    </links>
    <modins_calls>
      <example>id</example>
      <example>call_stack_id</example>
      <example>modins</example>
      <example>props_set_id</example>
    </modins_calls>
    <props_sets>
      <example>id</example>
      <example>key</example>
      <example>value</example>
    </props_sets>
    <trafos>
      <example>id</example>
      <example>mic_id</example>
      <example>ord_nr</example>
      <example>desc</example>
      <example>config_file</example>
    </trafos>
    <ts_sets>
      <example>id</ex

## Execute query
Deze tool geeft alle mogelijke vrijheid: het taalmodel kan elke SQL query uitvoeren.

In [19]:
from talk_with_fews.tools.config.sql_query import execute_query
from talk_with_fews.models.sql_query import SQLQueryInput, SQLQueryOutput

sql_query_input = SQLQueryInput(
    query="SELECT * FROM endpoints limit 3",
)

sql_query_output: SQLQueryOutput = await execute_query(ctx=ctx, input=sql_query_input)

In [20]:
print(format_as_xml(sql_query_output, root_tag="SQLQueryOutput"))

<SQLQueryOutput>
  <results>
    <example>
      <tss_id>1</tss_id>
      <desc>filters</desc>
    </example>
    <example>
      <tss_id>2</tss_id>
      <desc>filters</desc>
    </example>
    <example>
      <tss_id>3</tss_id>
      <desc>filters</desc>
    </example>
  </results>
  <column_names>
    <example>tss_id</example>
    <example>desc</example>
  </column_names>
  <row_count>3</row_count>
  <query_executed>SELECT * FROM endpoints limit 3</query_executed>
</SQLQueryOutput>
